# Materials science basics with ASE

ASE (the Atomic Simulation Environment) builds crystal structures, attaches a calculator (a
model for the energy/forces), and runs geometry optimization - the same workflow real DFT
materials-science codes use, just with `EMT` here: a fast built-in classical potential for
metals, so no external DFT code or GPU is needed to see the workflow end to end.

In [ ]:
from ase.build import bulk
from ase.calculators.emt import EMT
import numpy as np

# Bulk copper, FCC structure, scanning the lattice constant to find the energy minimum -
# same idea as scanning bond length in the pyscf notebook, one level up in scale.
a_values = np.linspace(3.4, 3.8, 21)
energies = []
for a in a_values:
    atoms = bulk("Cu", "fcc", a=a)
    atoms.calc = EMT()
    energies.append(atoms.get_potential_energy())
energies = np.array(energies)

a_min = a_values[np.argmin(energies)]
print(f"EMT-computed equilibrium lattice constant: {a_min:.3f} A")
print(f"experimental Cu lattice constant:           3.615 A")
print()
print("Note: EMT is a fast, approximate classical potential, so a small gap from the real")
print("experimental value is expected - it's not a bug. Real DFT calculators (e.g. GPAW,")
print("Quantum ESPRESSO via ASE's calculator interface) get much closer, at far higher cost.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(a_values, energies, "o-")
plt.axvline(3.615, color="tab:red", linestyle="--", label="experimental (3.615 A)")
plt.axvline(a_min, color="tab:green", linestyle="--", label=f"EMT minimum ({a_min:.3f} A)")
plt.xlabel("lattice constant a (Angstrom)")
plt.ylabel("energy (eV)")
plt.title("Bulk Cu (FCC): energy vs. lattice constant")
plt.legend()
plt.show()

## Full cell relaxation

Instead of scanning by hand, let an optimizer find the equilibrium directly. A plain
atom-position relaxation can't move a symmetric bulk lattice constant at all - every atom's
force is zero by symmetry regardless of cell size. `FrechetCellFilter` wraps the atoms so the
optimizer can also adjust the cell itself to relieve internal stress, which is what actually
finds the equilibrium lattice constant.

In [ ]:
from ase.optimize import BFGS
from ase.filters import FrechetCellFilter

atoms = bulk("Cu", "fcc", a=3.5)  # deliberately off from equilibrium
atoms.calc = EMT()

opt = BFGS(FrechetCellFilter(atoms), logfile=None)
opt.run(fmax=0.01)

relaxed_a = atoms.cell.cellpar()[0] * 2 ** 0.5  # conventional-cell edge -> lattice constant a
print(f"relaxed lattice constant: {relaxed_a:.3f} A")
print(f"(compare to the {a_min:.3f} A found by the scan above - should be close)")

## Next steps

- Try a different EMT-supported metal, such as `bulk("Al", "fcc", a=4.05)` or
  `bulk("Ni", "fcc", a=3.52)`. ASE's EMT implementation supports Al, Cu, Ag, Au,
  Ni, Pd, and Pt as standard metals.
- Build a surface instead of bulk (`ase.build.fcc111`) and compute a surface energy.
- Swap `EMT()` for a real DFT calculator if you have one set up (e.g. GPAW) - the rest of the
  workflow (build structure, attach calculator, optimize) stays identical, which is the point
  of ASE's calculator abstraction.